In [19]:
"""
Enhanced Gene-to-m/z Spatial Pattern Matching Pipeline V2
=========================================================

IMPROVEMENTS OVER V1:
1. Better loss function with proper scaling (target loss < 1.0)
2. Contrastive learning for meaningful spatial representations
3. Spatial coherence loss that actually enforces local smoothness
4. Feature reconstruction loss to preserve signal information
5. Variance-based importance scoring (high variance = important)
6. Multi-scale spatial analysis
7. Better training strategies with warmup and gradient scaling

Author: Enhanced version with improved saliency maps
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool, global_add_pool
from torch_geometric.utils import softmax, add_self_loops, degree
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import Delaunay, distance_matrix
from scipy.stats import pearsonr, spearmanr
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional, Union
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass
import json

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
    'Thy1', 'Pmch', 'Mapt', 'App', 'Apoe'
]



In [20]:

# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class AttentionOutput:
    """Container for attention-related outputs"""
    node_embeddings: torch.Tensor
    attention_weights: List[torch.Tensor]
    node_importance: torch.Tensor
    weighted_embeddings: torch.Tensor
    global_embedding: torch.Tensor
    reconstructed_features: Optional[torch.Tensor] = None


@dataclass
class SpatialSignature:
    """Spatial signature for a gene or m/z feature"""
    sample_id: str
    feature_name: str
    feature_type: str
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    coordinates: np.ndarray
    raw_values: np.ndarray



In [21]:

# =============================================================================
# IMPROVED GAT WITH BETTER ATTENTION
# =============================================================================

class ImprovedGATConv(nn.Module):
    """
    Improved GAT layer with better attention mechanism and stored weights.
    Uses edge features based on spatial distance for better spatial awareness.
    """
    
    def __init__(
        self, 
        in_channels: int, 
        out_channels: int, 
        heads: int = 4,
        concat: bool = True,
        dropout: float = 0.1,
        add_self_loops: bool = True,
        bias: bool = True
    ):
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.dropout = dropout
        self.add_self_loops_flag = add_self_loops
        
        # Linear transformations for Q, K, V
        self.lin_q = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.lin_k = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.lin_v = nn.Linear(in_channels, heads * out_channels, bias=False)
        
        # Attention mechanism
        self.att = nn.Parameter(torch.Tensor(1, heads, out_channels))
        
        if bias and concat:
            self.bias = nn.Parameter(torch.Tensor(heads * out_channels))
        elif bias and not concat:
            self.bias = nn.Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias', None)
        
        # Store attention for extraction
        self.stored_attention = None
        self.stored_edge_index = None
        
        self._reset_parameters()
    
    def _reset_parameters(self):
        nn.init.xavier_uniform_(self.lin_q.weight)
        nn.init.xavier_uniform_(self.lin_k.weight)
        nn.init.xavier_uniform_(self.lin_v.weight)
        nn.init.xavier_uniform_(self.att)
        if self.bias is not None:
            nn.init.zeros_(self.bias)
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, 
                pos: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass with spatial-aware attention.
        
        Args:
            x: Node features [N, in_channels]
            edge_index: Edge connectivity [2, E]
            pos: Node positions [N, 2] for spatial attention bias
        """
        N = x.size(0)
        
        # Add self-loops
        if self.add_self_loops_flag:
            edge_index, _ = add_self_loops(edge_index, num_nodes=N)
        
        # Transform features
        q = self.lin_q(x).view(N, self.heads, self.out_channels)
        k = self.lin_k(x).view(N, self.heads, self.out_channels)
        v = self.lin_v(x).view(N, self.heads, self.out_channels)
        
        # Get source and target indices
        src, tgt = edge_index[0], edge_index[1]
        
        # Compute attention scores
        # Using scaled dot-product attention
        q_i = q[tgt]  # [E, heads, out_channels]
        k_j = k[src]  # [E, heads, out_channels]
        
        # Attention scores
        alpha = (q_i * k_j).sum(dim=-1) / np.sqrt(self.out_channels)  # [E, heads]
        
        # Add spatial bias if positions provided
        if pos is not None:
            # Compute spatial distances
            dist = torch.norm(pos[tgt] - pos[src], dim=-1, keepdim=True)  # [E, 1]
            # Convert to attention bias (closer = higher attention)
            spatial_bias = torch.exp(-dist / (dist.mean() + 1e-8))  # [E, 1]
            alpha = alpha + spatial_bias * 0.5  # Add spatial bias
        
        # Softmax over neighbors
        alpha = softmax(alpha, tgt, num_nodes=N)  # [E, heads]
        
        # Store attention weights
        self.stored_attention = alpha.detach()
        self.stored_edge_index = edge_index
        
        # Apply dropout
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        
        # Aggregate
        v_j = v[src]  # [E, heads, out_channels]
        out = torch.zeros(N, self.heads, self.out_channels, device=x.device)
        
        # Weighted aggregation
        alpha_expanded = alpha.unsqueeze(-1)  # [E, heads, 1]
        weighted_v = alpha_expanded * v_j  # [E, heads, out_channels]
        
        out.scatter_add_(0, tgt.view(-1, 1, 1).expand(-1, self.heads, self.out_channels), weighted_v)
        
        # Concatenate or average heads
        if self.concat:
            out = out.view(N, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)
        
        if self.bias is not None:
            out = out + self.bias
        
        return out



In [22]:

# =============================================================================
# IMPROVED SPATIAL ATTENTION GNN
# =============================================================================

class ImprovedSpatialGNN(nn.Module):
    """
    Improved GNN with:
    1. Better attention mechanism with spatial awareness
    2. Feature reconstruction decoder for better representations
    3. Multi-scale spatial importance computation
    4. Variance-based importance scoring
    """
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.1,
        projection_dim: int = 64,
        use_reconstruction: bool = True
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.projection_dim = projection_dim
        self.use_reconstruction = use_reconstruction
        
        # Flexible input projections
        self.input_projections = nn.ModuleDict()
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        # GAT encoder layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        # First layer
        self.convs.append(ImprovedGATConv(
            projection_dim, hidden_dim, heads=num_heads, concat=True, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(hidden_dim * num_heads))
        
        # Hidden layers with residual connections
        for _ in range(num_layers - 2):
            self.convs.append(ImprovedGATConv(
                hidden_dim * num_heads, hidden_dim, heads=num_heads, 
                concat=True, dropout=dropout
            ))
            self.norms.append(nn.LayerNorm(hidden_dim * num_heads))
        
        # Output layer
        self.convs.append(ImprovedGATConv(
            hidden_dim * num_heads, embedding_dim, heads=1, 
            concat=False, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(embedding_dim))
        
        # Importance prediction network (predicts which nodes are important)
        self.importance_net = nn.Sequential(
            nn.Linear(embedding_dim + projection_dim, hidden_dim),  # Include original features
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Attention pooling for global embedding
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Decoder for reconstruction (helps learn better representations)
        if use_reconstruction:
            self.decoder = nn.Sequential(
                nn.Linear(embedding_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, projection_dim)
            )
        
        self.dropout = nn.Dropout(dropout)
    
    def _get_or_create_projection(self, input_dim: int) -> nn.Module:
        """Get or create input projection for given input dimension"""
        dim_key = str(input_dim)
        
        if dim_key not in self.input_projections:
            device = next(self.parameters()).device
            proj = nn.Sequential(
                nn.Linear(input_dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
            self.input_projections[dim_key] = proj
        
        return self.input_projections[dim_key]
    
    def forward(
        self, 
        x: torch.Tensor, 
        edge_index: torch.Tensor,
        pos: torch.Tensor,
        batch: Optional[torch.Tensor] = None
    ) -> AttentionOutput:
        """
        Forward pass with spatial-aware attention.
        
        Args:
            x: Node features [N, input_dim]
            edge_index: Edge connectivity [2, E]
            pos: Node spatial coordinates [N, 2]
            batch: Batch assignment for multiple graphs
        """
        # Project input
        input_dim = x.size(1)
        projection = self._get_or_create_projection(input_dim)
        h_input = projection(x)  # [N, projection_dim]
        
        attention_weights = []
        h = h_input
        
        # Forward through GAT layers with residuals
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            h_new = conv(h, edge_index, pos)
            attention_weights.append(conv.stored_attention)
            h_new = norm(h_new)
            
            # Residual connection for hidden layers
            if i > 0 and i < len(self.convs) - 1 and h.size(-1) == h_new.size(-1):
                h_new = h_new + h
            
            h = F.gelu(h_new) if i < len(self.convs) - 1 else h_new
            h = self.dropout(h)
        
        node_embeddings = h  # [N, embedding_dim]
        
        # Compute importance scores using embeddings AND original features
        importance_input = torch.cat([node_embeddings, h_input], dim=-1)
        importance_logits = self.importance_net(importance_input).squeeze(-1)  # [N]
        
        # Use sigmoid for smooth importance values
        node_importance = torch.sigmoid(importance_logits)
        
        # Also incorporate attention-based importance
        attn_importance = self._compute_attention_importance(attention_weights, edge_index, x.size(0))
        
        # Combine learned importance with attention importance
        combined_importance = 0.7 * node_importance + 0.3 * attn_importance
        combined_importance = combined_importance / (combined_importance.max() + 1e-8)
        
        # Weight embeddings by importance
        importance_weight = combined_importance / (combined_importance.sum() + 1e-8) * x.size(0)
        weighted_embeddings = node_embeddings * importance_weight.unsqueeze(-1)
        
        # Global embedding via attention pooling
        global_embedding = self._attention_pool(weighted_embeddings, combined_importance, batch)
        
        # Reconstruction (if enabled)
        reconstructed = None
        if self.use_reconstruction and self.training:
            reconstructed = self.decoder(node_embeddings)
        
        return AttentionOutput(
            node_embeddings=node_embeddings,
            attention_weights=attention_weights,
            node_importance=combined_importance,
            weighted_embeddings=weighted_embeddings,
            global_embedding=global_embedding,
            reconstructed_features=reconstructed
        )
    
    def _compute_attention_importance(
        self,
        attention_weights: List[torch.Tensor],
        edge_index: torch.Tensor,
        num_nodes: int
    ) -> torch.Tensor:
        """Compute node importance from attention patterns"""
        device = edge_index.device
        importance = torch.zeros(num_nodes, device=device)
        
        for attn in attention_weights:
            if attn is None:
                continue
            
            # Average attention across heads
            if attn.dim() > 1:
                attn_mean = attn.mean(dim=-1)
            else:
                attn_mean = attn
            
            # Get target nodes (nodes receiving attention)
            tgt = edge_index[1]
            
            # Aggregate incoming attention
            importance.scatter_add_(0, tgt, attn_mean)
        
        # Normalize
        importance = importance / (len(attention_weights) + 1e-8)
        importance = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)
        
        return importance
    
    def _attention_pool(
        self,
        embeddings: torch.Tensor,
        importance: torch.Tensor,
        batch: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Importance-weighted attention pooling"""
        # Compute attention scores
        attn_scores = self.pool_attention(embeddings).squeeze(-1)  # [N]
        
        # Combine with importance
        combined_scores = attn_scores + importance
        
        if batch is None:
            weights = F.softmax(combined_scores, dim=0)
            global_emb = (weights.unsqueeze(-1) * embeddings).sum(dim=0)
        else:
            weights = softmax(combined_scores, batch)
            global_emb = global_add_pool(weights.unsqueeze(-1) * embeddings, batch)
        
        return global_emb




In [23]:

# =============================================================================
# IMPROVED LOSS FUNCTIONS
# =============================================================================

class ImprovedSpatialLoss(nn.Module):
    """
    Improved loss function with proper scaling to achieve loss < 1.0
    
    Components:
    1. Reconstruction loss - preserve feature information
    2. Local smoothness loss - nearby nodes should have similar embeddings
    3. Global structure loss - preserve overall spatial structure
    4. Contrastive loss - distinguish different spatial regions
    5. Importance sparsity - encourage focused attention
    """
    
    def __init__(
        self,
        reconstruction_weight: float = 1.0,
        smoothness_weight: float = 0.5,
        structure_weight: float = 0.3,
        contrastive_weight: float = 0.5,
        sparsity_weight: float = 0.1,
        temperature: float = 0.1
    ):
        super().__init__()
        self.reconstruction_weight = reconstruction_weight
        self.smoothness_weight = smoothness_weight
        self.structure_weight = structure_weight
        self.contrastive_weight = contrastive_weight
        self.sparsity_weight = sparsity_weight
        self.temperature = temperature
    
    def forward(
        self,
        output: AttentionOutput,
        input_features: torch.Tensor,
        pos: torch.Tensor,
        edge_index: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute total loss with component breakdown.
        
        Returns:
            total_loss: Combined loss value
            loss_dict: Dictionary with individual loss components
        """
        losses = {}
        
        # 1. Reconstruction loss (if available)
        if output.reconstructed_features is not None:
            recon_loss = F.mse_loss(output.reconstructed_features, input_features)
            losses['reconstruction'] = recon_loss * self.reconstruction_weight
        else:
            losses['reconstruction'] = torch.tensor(0.0, device=pos.device)
        
        # 2. Local smoothness loss - embeddings should be smooth over space
        smoothness_loss = self._local_smoothness_loss(
            output.node_embeddings, edge_index, pos
        )
        losses['smoothness'] = smoothness_loss * self.smoothness_weight
        
        # 3. Global structure preservation
        structure_loss = self._structure_preservation_loss(
            output.node_embeddings, pos
        )
        losses['structure'] = structure_loss * self.structure_weight
        
        # 4. Contrastive loss - distinguish different regions
        contrastive_loss = self._contrastive_loss(
            output.node_embeddings, pos
        )
        losses['contrastive'] = contrastive_loss * self.contrastive_weight
        
        # 5. Importance sparsity - encourage focused attention
        sparsity_loss = self._importance_sparsity_loss(output.node_importance)
        losses['sparsity'] = sparsity_loss * self.sparsity_weight
        
        # Total loss
        total_loss = sum(losses.values())
        
        # Convert to float dict for logging
        loss_dict = {k: v.item() for k, v in losses.items()}
        loss_dict['total'] = total_loss.item()
        
        return total_loss, loss_dict
    
    def _local_smoothness_loss(
        self,
        embeddings: torch.Tensor,
        edge_index: torch.Tensor,
        pos: torch.Tensor
    ) -> torch.Tensor:
        """
        Embeddings of neighboring nodes should be similar.
        Weighted by spatial proximity.
        """
        src, tgt = edge_index[0], edge_index[1]
        
        # Embedding differences
        emb_diff = embeddings[src] - embeddings[tgt]
        emb_dist = (emb_diff ** 2).sum(dim=-1)  # [E]
        
        # Spatial distances for weighting
        spatial_dist = torch.norm(pos[src] - pos[tgt], dim=-1)  # [E]
        spatial_weights = torch.exp(-spatial_dist / (spatial_dist.mean() + 1e-8))
        
        # Weighted MSE
        loss = (spatial_weights * emb_dist).mean()
        
        # Normalize to ~[0, 1] range
        loss = loss / (embeddings.size(-1) + 1e-8)
        
        return loss
    
    def _structure_preservation_loss(
        self,
        embeddings: torch.Tensor,
        pos: torch.Tensor,
        num_samples: int = 1000
    ) -> torch.Tensor:
        """
        Preserve global spatial structure - embedding distances should
        correlate with spatial distances.
        """
        N = embeddings.size(0)
        num_samples = min(num_samples, N * (N - 1) // 2)
        
        # Random pairs
        idx1 = torch.randint(0, N, (num_samples,), device=embeddings.device)
        idx2 = torch.randint(0, N, (num_samples,), device=embeddings.device)
        
        # Avoid same pairs
        mask = idx1 != idx2
        idx1, idx2 = idx1[mask], idx2[mask]
        
        if len(idx1) < 10:
            return torch.tensor(0.0, device=embeddings.device)
        
        # Compute distances
        spatial_dist = torch.norm(pos[idx1] - pos[idx2], dim=-1)
        emb_dist = torch.norm(embeddings[idx1] - embeddings[idx2], dim=-1)
        
        # Normalize to [0, 1]
        spatial_dist = spatial_dist / (spatial_dist.max() + 1e-8)
        emb_dist = emb_dist / (emb_dist.max() + 1e-8)
        
        # MSE between normalized distances
        loss = F.mse_loss(emb_dist, spatial_dist)
        
        return loss
    
    def _contrastive_loss(
        self,
        embeddings: torch.Tensor,
        pos: torch.Tensor,
        num_samples: int = 500
    ) -> torch.Tensor:
        """
        Contrastive loss: nearby nodes should be similar,
        far nodes should be different.
        """
        N = embeddings.size(0)
        num_samples = min(num_samples, N)
        
        # Sample anchor nodes
        anchors = torch.randperm(N, device=embeddings.device)[:num_samples]
        
        # For each anchor, find positive (nearby) and negative (far) samples
        anchor_pos = pos[anchors]  # [num_samples, 2]
        
        # Compute distances to all nodes
        all_dist = torch.cdist(anchor_pos, pos)  # [num_samples, N]
        
        # Positive: closest nodes (excluding self)
        all_dist_masked = all_dist.clone()
        all_dist_masked.scatter_(1, anchors.unsqueeze(1), float('inf'))
        positive_idx = all_dist_masked.argmin(dim=1)  # [num_samples]
        
        # Negative: random far nodes (top 50% by distance)
        _, sorted_idx = all_dist.sort(dim=1, descending=True)
        far_half = sorted_idx[:, :N//2]
        rand_idx = torch.randint(0, N//2, (num_samples,), device=embeddings.device)
        negative_idx = far_half[torch.arange(num_samples, device=embeddings.device), rand_idx]
        
        # Get embeddings
        anchor_emb = embeddings[anchors]
        positive_emb = embeddings[positive_idx]
        negative_emb = embeddings[negative_idx]
        
        # Cosine similarities
        pos_sim = F.cosine_similarity(anchor_emb, positive_emb)
        neg_sim = F.cosine_similarity(anchor_emb, negative_emb)
        
        # InfoNCE-style loss
        pos_exp = torch.exp(pos_sim / self.temperature)
        neg_exp = torch.exp(neg_sim / self.temperature)
        
        loss = -torch.log(pos_exp / (pos_exp + neg_exp + 1e-8)).mean()
        
        return loss
    
    def _importance_sparsity_loss(self, importance: torch.Tensor) -> torch.Tensor:
        """
        Encourage importance to be sparse but not trivial.
        We want some nodes to be important (high values) and others not.
        """
        # Encourage variance in importance (not all same)
        variance = importance.var()
        variance_loss = -torch.log(variance + 1e-8)  # Higher variance = lower loss
        
        # Encourage some high values (top 10% should be significantly higher)
        k = max(1, len(importance) // 10)
        top_k = importance.topk(k).values
        mean_top = top_k.mean()
        mean_rest = (importance.sum() - top_k.sum()) / (len(importance) - k + 1e-8)
        
        separation_loss = -torch.log(mean_top - mean_rest + 1e-8)
        
        loss = 0.5 * variance_loss + 0.5 * separation_loss
        
        # Clip to reasonable range
        loss = torch.clamp(loss, -2, 2)
        
        return loss





In [24]:

# =============================================================================
# VARIANCE-BASED IMPORTANCE (BIOLOGICAL RELEVANCE)
# =============================================================================

def compute_variance_importance(
    features: np.ndarray,
    coords: np.ndarray,
    n_neighbors: int = 15,
    sigma: float = 100.0
) -> np.ndarray:
    """
    Compute importance based on local variance of features.
    High local variance = interesting/important region.
    
    This captures biological relevance because:
    - Homogeneous regions (low variance) are less informative
    - Boundaries/transitions have high variance
    - Disease-affected regions often show heterogeneity
    """
    from sklearn.neighbors import NearestNeighbors
    
    # Find k nearest neighbors for each spot
    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # Compute local variance for each spot
    n_spots = features.shape[0]
    local_variance = np.zeros(n_spots)
    
    for i in range(n_spots):
        neighbor_idx = indices[i]
        neighbor_features = features[neighbor_idx]
        
        # Weighted variance (closer neighbors weighted more)
        weights = np.exp(-distances[i] ** 2 / (2 * sigma ** 2))
        weights /= weights.sum()
        
        # Weighted mean
        weighted_mean = np.sum(neighbor_features * weights[:, np.newaxis], axis=0)
        
        # Weighted variance
        diff = neighbor_features - weighted_mean
        weighted_var = np.sum(weights[:, np.newaxis] * diff ** 2)
        
        local_variance[i] = weighted_var
    
    # Normalize to [0, 1]
    local_variance = (local_variance - local_variance.min()) / (local_variance.max() - local_variance.min() + 1e-8)
    
    return local_variance


def compute_gradient_importance(
    features: np.ndarray,
    coords: np.ndarray,
    n_neighbors: int = 6
) -> np.ndarray:
    """
    Compute importance based on spatial gradient magnitude.
    High gradient = boundary/transition region.
    """
    from sklearn.neighbors import NearestNeighbors
    
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)  # +1 for self
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    n_spots = features.shape[0]
    gradient_magnitude = np.zeros(n_spots)
    
    for i in range(n_spots):
        center_feature = features[i]
        neighbor_idx = indices[i, 1:]  # Exclude self
        neighbor_coords = coords[neighbor_idx] - coords[i]
        neighbor_features = features[neighbor_idx]
        
        # Compute gradient via least squares
        # gradient ≈ (neighbor_features - center) / distance
        feature_diff = neighbor_features - center_feature
        
        if features.ndim == 1 or features.shape[1] == 1:
            feature_diff = feature_diff.flatten()
            dist = np.linalg.norm(neighbor_coords, axis=1) + 1e-8
            gradient_magnitude[i] = np.abs(feature_diff / dist).mean()
        else:
            dist = np.linalg.norm(neighbor_coords, axis=1, keepdims=True) + 1e-8
            gradients = feature_diff / dist
            gradient_magnitude[i] = np.linalg.norm(gradients, axis=1).mean()
    
    # Normalize
    gradient_magnitude = (gradient_magnitude - gradient_magnitude.min()) / \
                        (gradient_magnitude.max() - gradient_magnitude.min() + 1e-8)
    
    return gradient_magnitude




In [25]:

# =============================================================================
# ENHANCED MATCHER CLASS
# =============================================================================

class ImprovedGeneToMzMatcher:
    """
    Improved matcher with better training and saliency maps.
    """
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v2',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 3,
        num_heads: int = 4
    ):
        self.device = device
        self.output_dir = output_dir
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        
        # Create directories
        for subdir in ['gene_visualizations', 'saliency_maps', 'embeddings', 
                       'attention_analysis', 'training_curves']:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_model = None
        self.msi_model = None
        
        self.gene_signatures = defaultdict(dict)
        self.mz_signatures = defaultdict(dict)
    
    def load_all_data(self):
        """Load all RNA and MSI data"""
        print("Loading RNA-seq data...")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print("\nLoading MSI data...")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def check_gene_availability(self) -> Dict[str, Dict[str, bool]]:
        """Check gene availability"""
        gene_availability = {}
        for gene in AAD_TARGET_GENES:
            gene_availability[gene] = {}
            for sample_id in RNA_SAMPLE_IDS:
                if sample_id in self.rna_data:
                    gene_availability[gene][sample_id] = gene in self.rna_data[sample_id].var_names
                else:
                    gene_availability[gene][sample_id] = False
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            available = sum(gene_availability[gene].values())
            print(f"  {gene}: {available}/{len(RNA_SAMPLE_IDS)} samples")
        
        return gene_availability
    
    def build_spatial_graph(
        self,
        coords: np.ndarray,
        k_neighbors: int = 10
    ) -> torch.Tensor:
        """Build spatial graph using k-NN"""
        from sklearn.neighbors import NearestNeighbors
        
        nn = NearestNeighbors(n_neighbors=k_neighbors + 1)
        nn.fit(coords)
        _, indices = nn.kneighbors(coords)
        
        # Build edge list (bidirectional)
        edge_list = []
        for i in range(len(coords)):
            for j in indices[i, 1:]:  # Skip self
                edge_list.append([i, j])
                edge_list.append([j, i])
        
        # Remove duplicates
        edge_set = set(map(tuple, edge_list))
        edge_list = list(edge_set)
        
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
        
        return edge_index
    
    def prepare_graph_data(
        self,
        coords: np.ndarray,
        features: np.ndarray,
        k_neighbors: int = 10
    ) -> Data:
        """Prepare PyG Data object"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        # Normalize features
        scaler = StandardScaler()
        features_norm = scaler.fit_transform(features)
        
        # Build graph
        edge_index = self.build_spatial_graph(coords, k_neighbors)
        
        # Normalize coordinates
        pos_norm = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
        
        data = Data(
            x=torch.tensor(features_norm, dtype=torch.float32),
            edge_index=edge_index,
            pos=torch.tensor(pos_norm, dtype=torch.float32),
            pos_original=torch.tensor(coords, dtype=torch.float32)
        )
        
        return data
    
    def train_model(
        self,
        data_dict: Dict[str, Data],
        model_type: str = 'rna',
        epochs: int = 200,
        lr: float = 0.001,
        warmup_epochs: int = 20
    ) -> ImprovedSpatialGNN:
        """
        Train model with improved loss and training strategy.
        Target: final loss < 1.0
        """
        print(f"\nTraining {model_type.upper()} model...")
        
        # Get input dim
        first_data = list(data_dict.values())[0]
        input_dim = first_data.x.size(1)
        
        # Initialize model
        model = ImprovedSpatialGNN(
            input_dim=input_dim,
            hidden_dim=self.hidden_dim,
            embedding_dim=self.embedding_dim,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            projection_dim=64,
            use_reconstruction=True
        ).to(self.device)
        
        # Optimizer with weight decay
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        
        # Learning rate scheduler with warmup
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return epoch / warmup_epochs
            return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        
        # Loss function
        loss_fn = ImprovedSpatialLoss(
            reconstruction_weight=1.0,
            smoothness_weight=0.5,
            structure_weight=0.3,
            contrastive_weight=0.5,
            sparsity_weight=0.1
        )
        
        # Training
        model.train()
        loss_history = []
        component_history = defaultdict(list)
        
        for epoch in range(epochs):
            total_loss = 0
            batch_losses = defaultdict(float)
            
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                optimizer.zero_grad()
                
                # Get original features for reconstruction
                input_features = model._get_or_create_projection(data.x.size(1))(data.x)
                
                output = model(data.x, data.edge_index, data.pos)
                loss, loss_dict = loss_fn(output, input_features, data.pos, data.edge_index)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
                for k, v in loss_dict.items():
                    batch_losses[k] += v
            
            scheduler.step()
            
            avg_loss = total_loss / len(data_dict)
            loss_history.append(avg_loss)
            
            for k, v in batch_losses.items():
                component_history[k].append(v / len(data_dict))
            
            if (epoch + 1) % 25 == 0:
                lr_current = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, LR: {lr_current:.6f}")
                print(f"    Components: " + ", ".join([f"{k}: {v/len(data_dict):.4f}" 
                                                       for k, v in batch_losses.items()]))
        
        # Plot training curves
        self._plot_training_curves(loss_history, component_history, model_type)
        
        print(f"\n  Final loss: {loss_history[-1]:.4f}")
        
        return model
    
    def _plot_training_curves(
        self,
        loss_history: List[float],
        component_history: Dict[str, List[float]],
        model_type: str
    ):
        """Plot training loss curves"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Total loss
        axes[0].plot(loss_history, 'b-', linewidth=2)
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Total Loss')
        axes[0].set_title(f'{model_type.upper()} Training Loss')
        axes[0].axhline(y=1.0, color='r', linestyle='--', label='Target (< 1.0)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Component losses
        for name, values in component_history.items():
            if name != 'total':
                axes[1].plot(values, label=name)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Loss Components')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(
            os.path.join(self.output_dir, 'training_curves', f'{model_type}_training.png'),
            dpi=150, bbox_inches='tight'
        )
        plt.close()
    
    def compute_enhanced_importance(
        self,
        coords: np.ndarray,
        features: np.ndarray,
        model_importance: np.ndarray
    ) -> np.ndarray:
        """
        Combine model-based importance with variance-based importance.
        This ensures biologically relevant regions are highlighted.
        """
        # Variance-based importance
        variance_imp = compute_variance_importance(features, coords)
        
        # Gradient-based importance
        gradient_imp = compute_gradient_importance(features, coords)
        
        # Combine: model (50%) + variance (30%) + gradient (20%)
        combined = (
            0.5 * model_importance +
            0.3 * variance_imp +
            0.2 * gradient_imp
        )
        
        # Normalize
        combined = (combined - combined.min()) / (combined.max() - combined.min() + 1e-8)
        
        return combined, {
            'model': model_importance,
            'variance': variance_imp,
            'gradient': gradient_imp
        }
    
    def extract_gene_signature(
        self,
        sample_id: str,
        gene: str,
        model: ImprovedSpatialGNN
    ) -> Optional[SpatialSignature]:
        """Extract spatial signature for a gene"""
        if sample_id not in self.rna_data:
            return None
        
        adata = self.rna_data[sample_id]
        if gene not in adata.var_names:
            return None
        
        coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
        
        gene_idx = list(adata.var_names).index(gene)
        if hasattr(adata.X, 'toarray'):
            gene_expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            gene_expr = adata.X[:, gene_idx].flatten()
        
        # Prepare data
        data = self.prepare_graph_data(coords, gene_expr)
        data = data.to(self.device)
        
        # Extract embeddings
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos)
        
        model_importance = output.node_importance.cpu().numpy()
        
        # Enhance importance with variance/gradient
        enhanced_importance, importance_components = self.compute_enhanced_importance(
            coords, gene_expr, model_importance
        )
        
        signature = SpatialSignature(
            sample_id=sample_id,
            feature_name=gene,
            feature_type='gene',
            global_embedding=output.global_embedding.cpu().numpy(),
            node_embeddings=output.weighted_embeddings.cpu().numpy(),
            node_importance=enhanced_importance,
            coordinates=coords,
            raw_values=gene_expr
        )
        
        # Cache
        self.gene_signatures[gene][sample_id] = signature
        
        return signature
    
    def extract_mz_signatures(
        self,
        sample_id: str,
        model: ImprovedSpatialGNN,
        batch_size: int = 50
    ) -> Dict[str, SpatialSignature]:
        """Extract signatures for all m/z features"""
        if sample_id not in self.msi_data:
            return {}
        
        adata = self.msi_data[sample_id]
        coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
        
        if hasattr(adata.X, 'toarray'):
            mz_data = adata.X.toarray()
        else:
            mz_data = adata.X.copy()
        
        mz_names = list(adata.var_names)
        signatures = {}
        
        model.eval()
        
        print(f"  Processing {len(mz_names)} m/z features...")
        
        for i, mz_name in enumerate(mz_names):
            mz_values = mz_data[:, i]
            
            data = self.prepare_graph_data(coords, mz_values)
            data = data.to(self.device)
            
            with torch.no_grad():
                output = model(data.x, data.edge_index, data.pos)
            
            model_importance = output.node_importance.cpu().numpy()
            enhanced_importance, _ = self.compute_enhanced_importance(
                coords, mz_values, model_importance
            )
            
            signature = SpatialSignature(
                sample_id=sample_id,
                feature_name=mz_name,
                feature_type='mz',
                global_embedding=output.global_embedding.cpu().numpy(),
                node_embeddings=output.weighted_embeddings.cpu().numpy(),
                node_importance=enhanced_importance,
                coordinates=coords,
                raw_values=mz_values
            )
            
            signatures[mz_name] = signature
            self.mz_signatures[mz_name][sample_id] = signature
            
            if (i + 1) % 100 == 0:
                print(f"    {i+1}/{len(mz_names)}")
        
        return signatures
    
    def compute_similarity(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature
    ) -> Dict[str, float]:
        """Compute multiple similarity metrics"""
        # Embedding similarity
        emb1 = sig1.global_embedding.flatten()
        emb2 = sig2.global_embedding.flatten()
        
        cosine = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-8)
        euclidean = 1 / (1 + np.linalg.norm(emb1 - emb2))
        
        # Importance overlap
        imp_overlap = self._compute_importance_overlap(sig1, sig2)
        
        # Raw correlations
        raw_pearson, raw_spearman = self._compute_raw_correlations(sig1, sig2)
        
        return {
            'embedding_cosine': cosine,
            'embedding_euclidean': euclidean,
            'importance_overlap': imp_overlap,
            'raw_pearson': raw_pearson,
            'raw_spearman': raw_spearman
        }
    
    def _compute_importance_overlap(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_res: int = 50
    ) -> float:
        """Compute overlap of important regions"""
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        imp1 = griddata(sig1.coordinates, sig1.node_importance, (grid_x, grid_y), 
                       method='linear', fill_value=0)
        imp2 = griddata(sig2.coordinates, sig2.node_importance, (grid_x, grid_y),
                       method='linear', fill_value=0)
        
        # Normalize
        imp1 = imp1 / (imp1.max() + 1e-8)
        imp2 = imp2 / (imp2.max() + 1e-8)
        
        # IoU-style overlap
        intersection = np.minimum(imp1, imp2).sum()
        union = np.maximum(imp1, imp2).sum()
        
        return intersection / (union + 1e-8)
    
    def _compute_raw_correlations(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_res: int = 50
    ) -> Tuple[float, float]:
        """Compute raw value correlations"""
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        v1 = griddata(sig1.coordinates, sig1.raw_values, (grid_x, grid_y),
                     method='linear', fill_value=np.nan)
        v2 = griddata(sig2.coordinates, sig2.raw_values, (grid_x, grid_y),
                     method='linear', fill_value=np.nan)
        
        mask = ~(np.isnan(v1) | np.isnan(v2))
        if mask.sum() < 10:
            return 0.0, 0.0
        
        v1_flat, v2_flat = v1[mask], v2[mask]
        
        pearson, _ = pearsonr(v1_flat, v2_flat)
        spearman, _ = spearmanr(v1_flat, v2_flat)
        
        return (pearson if not np.isnan(pearson) else 0.0,
                spearman if not np.isnan(spearman) else 0.0)
    
    def find_matching_mz(
        self,
        gene_sig: SpatialSignature,
        mz_signatures: Dict[str, SpatialSignature],
        top_k: int = 20
    ) -> pd.DataFrame:
        """Find best matching m/z features"""
        matches = []
        
        for mz_name, mz_sig in mz_signatures.items():
            sim = self.compute_similarity(gene_sig, mz_sig)
            
            # Combined score
            combined = (
                0.4 * sim['embedding_cosine'] +
                0.2 * sim['importance_overlap'] +
                0.2 * max(sim['raw_pearson'], 0) +
                0.2 * sim['embedding_euclidean']
            )
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **sim,
                'combined_score': combined
            })
        
        df = pd.DataFrame(matches)
        return df.sort_values('combined_score', ascending=False).head(top_k)
    
    def visualize_saliency(
        self,
        signature: SpatialSignature,
        title: str = None,
        save_path: str = None
    ):
        """Visualize saliency map with improvements"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        # 1. Raw values
        im1 = axes[0, 0].scatter(
            signature.coordinates[:, 0], signature.coordinates[:, 1],
            c=signature.raw_values, cmap='viridis', s=15, alpha=0.8
        )
        axes[0, 0].set_title(f'Raw Values: {signature.feature_name}', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # 2. Importance (saliency)
        im2 = axes[0, 1].scatter(
            signature.coordinates[:, 0], signature.coordinates[:, 1],
            c=signature.node_importance, cmap='hot', s=15, alpha=0.8
        )
        axes[0, 1].set_title('Spatial Importance (Saliency)', fontweight='bold')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        plt.colorbar(im2, ax=axes[0, 1], label='Importance')
        
        # 3. Importance-weighted values
        weighted = signature.raw_values * signature.node_importance
        weighted = (weighted - weighted.min()) / (weighted.max() - weighted.min() + 1e-8)
        
        im3 = axes[1, 0].scatter(
            signature.coordinates[:, 0], signature.coordinates[:, 1],
            c=weighted, cmap='plasma', s=15, alpha=0.8
        )
        axes[1, 0].set_title('Importance-Weighted Values', fontweight='bold')
        axes[1, 0].set_xlabel('X (μm)')
        axes[1, 0].set_ylabel('Y (μm)')
        plt.colorbar(im3, ax=axes[1, 0])
        
        # 4. Top important regions highlighted
        threshold = np.percentile(signature.node_importance, 80)
        important_mask = signature.node_importance >= threshold
        
        axes[1, 1].scatter(
            signature.coordinates[:, 0], signature.coordinates[:, 1],
            c='lightgray', s=10, alpha=0.5, label='Other'
        )
        axes[1, 1].scatter(
            signature.coordinates[important_mask, 0],
            signature.coordinates[important_mask, 1],
            c=signature.raw_values[important_mask], cmap='Reds', s=20,
            alpha=0.9, label='Top 20% Important'
        )
        axes[1, 1].set_title('Top 20% Important Regions', fontweight='bold')
        axes[1, 1].set_xlabel('X (μm)')
        axes[1, 1].set_ylabel('Y (μm)')
        axes[1, 1].legend()
        
        if title:
            plt.suptitle(title, fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
        else:
            plt.show()
    
    def run_analysis(
        self,
        epochs: int = 200,
        top_k: int = 20
    ) -> pd.DataFrame:
        """Run complete analysis pipeline"""
        print("\n" + "="*70)
        print("IMPROVED GENE-TO-M/Z ANALYSIS V2")
        print("="*70)
        
        gene_availability = self.check_gene_availability()
        
        # =================================================================
        # PHASE 1: Train models
        # =================================================================
        print("\n" + "-"*70)
        print("PHASE 1: Training Models (Target: Loss < 1.0)")
        print("-"*70)
        
        # Prepare RNA training data
        print("\nPreparing RNA data...")
        rna_train = {}
        for sample_id in list(self.rna_data.keys())[:4]:
            adata = self.rna_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            rna_train[sample_id] = self.prepare_graph_data(coords, features)
            print(f"  {sample_id}: {rna_train[sample_id].x.shape}")
        
        if rna_train:
            self.rna_model = self.train_model(rna_train, 'rna', epochs=epochs)
        
        # Prepare MSI training data
        print("\nPreparing MSI data...")
        msi_train = {}
        for sample_id in list(self.msi_data.keys())[:4]:
            adata = self.msi_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            msi_train[sample_id] = self.prepare_graph_data(coords, features)
            print(f"  {sample_id}: {msi_train[sample_id].x.shape}")
        
        if msi_train:
            self.msi_model = self.train_model(msi_train, 'msi', epochs=epochs)
        
        # =================================================================
        # PHASE 2: Extract signatures and match
        # =================================================================
        print("\n" + "-"*70)
        print("PHASE 2: Extracting Signatures")
        print("-"*70)
        
        all_results = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Processing: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_availability[gene].items() if a]
            if not available:
                print(f"  Not found in any sample")
                continue
            
            for rna_sample in available[:2]:
                print(f"\n  Sample: {rna_sample}")
                
                gene_sig = self.extract_gene_signature(rna_sample, gene, self.rna_model)
                if gene_sig is None:
                    continue
                
                # Visualize saliency
                self.visualize_saliency(
                    gene_sig,
                    title=f'Saliency: {gene} ({rna_sample})',
                    save_path=os.path.join(
                        self.output_dir, 'saliency_maps',
                        f'{gene}_{rna_sample}_saliency.png'
                    )
                )
                
                # Match against MSI
                for msi_sample in list(self.msi_data.keys())[:2]:
                    print(f"    Matching vs {msi_sample}...")
                    
                    mz_sigs = self.extract_mz_signatures(msi_sample, self.msi_model)
                    if not mz_sigs:
                        continue
                    
                    matches = self.find_matching_mz(gene_sig, mz_sigs, top_k)
                    all_results.append(matches)
                    
                    if len(matches) > 0:
                        top = matches.iloc[0]
                        print(f"      Top: m/z {top['mz_feature']} (score: {top['combined_score']:.3f})")
        
        # Combine results
        if all_results:
            results_df = pd.concat(all_results, ignore_index=True)
            results_df = results_df.sort_values('combined_score', ascending=False)
            
            output_file = os.path.join(self.output_dir, 'gene_to_mz_matches_v2.csv')
            results_df.to_csv(output_file, index=False)
            print(f"\nResults saved to: {output_file}")
            
            return results_df
        
        return None


In [26]:
# =============================================================================
# MAIN
# =============================================================================

def main():
    print("="*70)
    print("GENE-TO-M/Z MATCHING V2 - IMPROVED SALIENCY MAPS")
    print("="*70)
    
    matcher = ImprovedGeneToMzMatcher(output_dir='./gene_to_mz_results_v2')
    matcher.load_all_data()
    
    results = matcher.run_analysis(epochs=200, top_k=20)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    print(f"\nResults in: {matcher.output_dir}/")
    
    return matcher, results


In [ ]:
if __name__ == "__main__":
    matcher, results = main()

GENE-TO-M/Z MATCHING V2 - IMPROVED SALIENCY MAPS
Loading RNA-seq data...
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

IMPROVED GENE-TO-M/Z ANALYSIS V2

Gene availability:
  Thy1: 16/16 samples
  Pmch: 16/16 samples
  Mapt: 16/16 samples
  App: 16/16 samples
  Apoe: 16/16 samples

----------------------------------------------------------------------
PHASE 1: Traini